In [1]:
%cd ..

d:\OSAS\GenSet


In [10]:
# src/main.py
import argparse
import json
from src.GenSet.config import Config
from src.GenSet.dataset_generator import DatasetGenerator


def parse_args() -> argparse.Namespace:
    """Parse the same flags that the CLI example (2) uses."""
    parser = argparse.ArgumentParser(
        description="Generate a balanced, multilingual dataset."
    )
    parser.add_argument(
        "--num-samples",
        type=int,
        required=True,
        help="How many examples to generate.",
    )
    parser.add_argument(
        "--multilingual",
        action="store_true",
        help="Include multilingual prompts.",
    )
    parser.add_argument(
        "--balance-labels",
        action="store_true",
        help="Force an even split across the supplied labels.",
    )
    parser.add_argument(
        "--ollama-key-weights",
        type=str,
        default=None,
        help="Comma‑separated weights for the Ollama models (e.g. \"0.4,0.3\").",
    )
    parser.add_argument(
        "--models",
        type=str,
        required=True,
        help=(
            "JSON string describing the model lists. Example: "
            "'{\"ollama\": [\"gemma3:4b-cloud:0.6\", \"gpt-oss:20b-cloud\"],"
            " \"mistral\": [\"mistral-small-2506:0.4\", \"mistral-medium-latest\"]}'"
        ),
    )
    parser.add_argument(
        "--labels",
        type=str,
        default="positive,negative",
        help="Comma‑separated list of label names.",
    )
    parser.add_argument(
        "--delay",
        type=float,
        default=0.8,
        help="Delay (seconds) between API calls.",
    )
    parser.add_argument(
        "--temperature",
        type=float,
        default=0.75,
        help="Sampling temperature for the LLMs.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()

    # ------------------------------------------------------------------
    # 1️⃣ Load the model configuration supplied via the --models flag
    # ------------------------------------------------------------------
    model_cfg = json.loads(args.models)

    # If the user gave separate Ollama weights, split and attach them.
    if args.ollama_key_weights:
        weights = [float(w) for w in args.ollama_key_weights.split(",")]
        # Pair each weight with the corresponding Ollama entry.
        # This mirrors the “key‑weights” concept from the CLI example.
        ollama_models = model_cfg.get("ollama", [])
        if len(weights) != len(ollama_models):
            raise ValueError(
                "Number of Ollama weights does not match number of Ollama models."
            )
        model_cfg["ollama"] = [
            f"{model}:{weight}" for model, weight in zip(ollama_models, weights)
        ]

    Config.set_model_config(model_cfg)
    Config.print_config()

    # ------------------------------------------------------------------
    # 2️⃣ Build the generator with the remaining CLI options
    # ------------------------------------------------------------------
    generator = DatasetGenerator()

    generator.create_dataset(
        num_samples=args.num_samples,
        multilingual=args.multilingual,
        balance_labels=args.balance_labels,
        labels=args.labels.split(","),
        delay=args.delay,
        temperature=args.temperature,
    )


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] --num-samples NUM_SAMPLES [--multilingual]
                             [--balance-labels]
                             [--ollama-key-weights OLLAMA_KEY_WEIGHTS]
                             --models MODELS [--labels LABELS] [--delay DELAY]
                             [--temperature TEMPERATURE]
ipykernel_launcher.py: error: the following arguments are required: --num-samples, --models


SystemExit: 2

C:\Users\AI_PC\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
